# EN4554 Assignment 2 : CNNs

**Name:** Indunil W.A.K.  
**Index:** 220240G  
**GitHub:** https://github.com/indunil-k/CNNs_220240G.git  

Test data is reserved for Task 4.

## 0. Setup

We fix seeds, create a fixed 45,000/5,000 split, and compute normalization statistics from the training subset only.

In [1]:
import random
import numpy as np
import torch
import torchvision
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

SEED = 42
NUM_WORKERS = 0  # reliable on both laptops and Colab
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('Device:', device)

PyTorch: 2.14.1+cpu
Device: cpu


In [2]:
raw_transform = transforms.ToTensor()
raw_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=raw_transform)
generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(raw_train), generator=generator).tolist()
train_indices = permutation[:45000]
val_indices = permutation[45000:]

stats_loader = DataLoader(Subset(raw_train, train_indices), batch_size=512, shuffle=False, num_workers=NUM_WORKERS)
channel_sum = torch.zeros(3)
channel_squared_sum = torch.zeros(3)
pixel_count = 0
for images, _ in stats_loader:
    images = images.float()
    channel_sum += images.sum(dim=(0, 2, 3))
    channel_squared_sum += (images ** 2).sum(dim=(0, 2, 3))
    pixel_count += images.shape[0] * images.shape[2] * images.shape[3]
mean = channel_sum / pixel_count
std = torch.sqrt(channel_squared_sum / pixel_count - mean ** 2)
print('Training mean:', mean)
print('Training std:', std)
print('Train size:', len(train_indices), 'Validation size:', len(val_indices))

Training mean: tensor([0.4917, 0.4823, 0.4467])
Training std: tensor([0.2471, 0.2435, 0.2616])
Train size: 45000 Validation size: 5000


In [3]:
normalize = transforms.Normalize(mean=mean.tolist(), std=std.tolist())
transform = transforms.Compose([transforms.ToTensor(), normalize])
train_dataset_full = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=transform)
train_dataset = Subset(train_dataset_full, train_indices)
val_dataset = Subset(train_dataset_full, val_indices)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print('Setup complete. Test data remains unused until Task 4.')

Setup complete. Test data remains unused until Task 4.


### Setup interpretation

The split is fixed by the seed, so all experiments can be compared fairly. Normalization uses only training statistics, preventing validation or test information from entering preprocessing.